# Notebook 2: Run Hardware Inference Benchmark

This notebook runs the fixed BFCL subset on one hardware backend.

It records:
- raw model output
- latency
- token counts
- tokens/sec
- GPU memory usage
- errors

The same notebook is used on each accelerator by changing the config file.

In [2]:
!git clone https://github.com/joerosh/BFCL-hardware-efficiency-study.git
%cd BFCL-hardware-efficiency-study

Cloning into 'BFCL-hardware-efficiency-study'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 32 (delta 7), reused 31 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 47.02 KiB | 5.88 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/BFCL-hardware-efficiency-study


In [3]:
!pip install -r requirements.txt
!pip install accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.3 MB/s eta 0:00:00


In [ ]:
import json
import os
import time
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from openai import OpenAI

In [ ]:
DATA_DIR = Path("../data")
CONFIG_DIR = Path("../configs")
RESULTS_DIR = Path("../results/raw")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TASK_PATH = DATA_DIR / "bfcl_subset.json"

# CONFIG_PATH = CONFIG_DIR / "t4_config.json"          # old Qwen T4 reference
# CONFIG_PATH = CONFIG_DIR / "t4_llama_config.json"    # main T4 Llama run
CONFIG_PATH = CONFIG_DIR / "groq_llama_config.json"    # Groq LPU run

In [6]:
def load_config(path):
    with open(path, "r", encoding="utf-8") as f:
        config = json.load(f)

    if "inherits" in config:
        parent_path = CONFIG_DIR / config["inherits"]
        with open(parent_path, "r", encoding="utf-8") as f:
            parent = json.load(f)

        parent.update({k: v for k, v in config.items() if k != "inherits"})
        return parent

    return config

config = load_config(CONFIG_PATH)
config

{'model_name': 'Qwen/Qwen2.5-7B-Instruct',
 'temperature': 0.0,
 'max_new_tokens': 256,
 'batch_size': 1,
 'num_repeats': 1,
 'device': 'cuda',
 'dtype': 'auto',
 'backend': 'transformers',
 'hardware': 'T4'}

In [ ]:
BACKEND = config.get("backend", "transformers_gpu")

print("Backend:", BACKEND)
print("Hardware:", config["hardware"])
print("Model:", config["model_name"])
print("Max new tokens:", config["max_new_tokens"])

In [7]:
with open(TASK_PATH, "r", encoding="utf-8") as f:
    tasks = json.load(f)

print(f"Loaded {len(tasks)} BFCL tasks.")
print("Categories:", pd.Series([t["category"] for t in tasks]).value_counts().to_dict())
print("First task ID:", tasks[0]["task_id"])

Loaded 100 BFCL tasks.
Categories: {'simple': 25, 'multiple': 25, 'parallel': 25, 'parallel_multiple': 25}
First task ID: simple_361


## Hardware Check

In [ ]:
if BACKEND == "transformers_gpu":
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("CUDA capability:", torch.cuda.get_device_capability(0))
        print("Total memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    else:
        print("No CUDA GPU detected.")
else:
    print(f"Skipping local GPU check for backend: {BACKEND}")

CUDA available: True
GPU: Tesla T4
CUDA capability: (7, 5)
Total memory GB: 14.56


## Load Model and Tokenizer

In [ ]:
BACKEND = config.get("backend", "transformers_gpu")
model_name = config["model_name"]

tokenizer = None
model = None
groq_client = None

if BACKEND == "transformers_gpu":
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    model_kwargs = {
        "torch_dtype": dtype,
        "device_map": "auto",
        "trust_remote_code": True
    }

    if config.get("load_in_8bit", False):
        model_kwargs["load_in_8bit"] = True
        model_kwargs.pop("torch_dtype", None)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        **model_kwargs
    )

    model.eval()
    print("Loaded Transformers GPU model:", model_name)

elif BACKEND == "groq_prompt_api":
    groq_api_key = os.environ.get("GROQ_API_KEY")

    if groq_api_key is None:
        raise ValueError("GROQ_API_KEY is not set. Set it in Colab secrets or environment variables.")

    groq_client = OpenAI(
        base_url="https://api.groq.com/openai/v1",
        api_key=groq_api_key
    )

    print("Configured Groq client:", model_name)

else:
    raise ValueError(f"Unsupported backend: {BACKEND}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: Qwen/Qwen2.5-7B-Instruct


# Warmup

In [ ]:
if BACKEND == "transformers_gpu":
    print("Running warmup...")

    warmup_prompt = "Hello"
    warmup_inputs = tokenizer(warmup_prompt, return_tensors="pt").to(model.device)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with torch.no_grad():
        _ = model.generate(
            **warmup_inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print("Warmup complete.")
else:
    print(f"Skipping warmup for backend: {BACKEND}")

## Prompt Formatting

BFCL prompts are stored in chat-message format. Each task also includes function/tool schemas. We use the model tokenizer's chat template where possible.

In [ ]:
def normalize_messages(prompt):
    """
    BFCL stores question as nested chat turns, often like:
    [[{"role": "user", "content": "..."}]]

    This function flattens it into:
    [{"role": "user", "content": "..."}]
    """
    if isinstance(prompt, list) and len(prompt) > 0:
        if isinstance(prompt[0], list):
            return prompt[0]
        if isinstance(prompt[0], dict):
            return prompt

    return [{"role": "user", "content": str(prompt)}]


def build_tool_prompt(task):
    """
    Shared prompt format for API and fallback cases.
    This keeps Groq output compatible with the existing Notebook 3 parser.
    """
    messages = normalize_messages(task["prompt"])
    user_content = messages[-1]["content"]
    tools = task["functions_or_tools"]

    system_content = (
        "You are a function-calling assistant.\n"
        "You must choose the correct function or functions from the available tools.\n"
        "Respond ONLY with tool call blocks in this exact format:\n"
        "<tool_call>\n"
        "{\"name\": \"function_name\", \"arguments\": {\"arg\": \"value\"}}\n"
        "</tool_call>\n\n"
        "If multiple calls are needed, output multiple <tool_call> blocks.\n\n"
        f"Available tools:\n{json.dumps(tools, indent=2)}"
    )

    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content}
    ]


def format_prompt(task):
    if BACKEND == "transformers_gpu":
        messages = normalize_messages(task["prompt"])
        tools = task["functions_or_tools"]

        try:
            return tokenizer.apply_chat_template(
                messages,
                tools=tools,
                tokenize=False,
                add_generation_prompt=True
            )
        except Exception:
            prompt_messages = build_tool_prompt(task)
            return tokenizer.apply_chat_template(
                prompt_messages,
                tokenize=False,
                add_generation_prompt=True
            )

    elif BACKEND == "groq_prompt_api":
        return build_tool_prompt(task)

    else:
        raise ValueError(f"Unsupported backend: {BACKEND}")

In [12]:
sample_prompt = format_prompt(tasks[0])
print(sample_prompt[:3000])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"name": "restaurant_finder", "description": "Locate restaurants based on certain criteria such as cuisine, city, and dietary preferences.", "parameters": {"type": "dict", "properties": {"city": {"type": "string", "description": "City where you are looking for the restaurant."}, "cuisine": {"type": "string", "description": "Type of cuisine you are interested in."}, "diet": {"type": "string", "description": "Dietary preferences. e.g. 'Vegetarian', 'Gluten-free', etc. Default 'Vegetarian'."}}, "required": ["city", "cuisine"]}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|

## Inference Function

This function runs one task, measures latency, counts tokens, and records GPU memory.

In [ ]:
def run_one_task(task, run_index=0):
    result = {
        "task_id": task["task_id"],
        "category": task["category"],
        "hardware": config["hardware"],
        "model": config["model_name"],
        "backend": config.get("backend", "transformers_gpu"),
        "run_index": run_index,
        "raw_output": None,
        "latency_s": None,
        "input_tokens": None,
        "output_tokens": None,
        "tokens_per_second": None,
        "peak_memory_mb": None,
        "peak_memory_allocated_mb": None,
        "peak_memory_reserved_mb": None,
        "error": None,
        "timestamp": datetime.now(timezone.utc).isoformat()
    }

    try:
        if BACKEND == "transformers_gpu":
            prompt_text = format_prompt(task)
            inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
            input_tokens = inputs["input_ids"].shape[-1]

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()
                torch.cuda.synchronize()

            start = time.perf_counter()

            with torch.no_grad():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=config["max_new_tokens"],
                    do_sample=False,
                    temperature=None,
                    pad_token_id=tokenizer.eos_token_id
                )

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            latency_s = time.perf_counter() - start

            generated_ids = output_ids[0][input_tokens:]
            output_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

            output_tokens = generated_ids.shape[-1]
            tokens_per_second = output_tokens / latency_s if latency_s > 0 else None

            peak_memory_mb = None
            peak_memory_allocated_mb = None
            peak_memory_reserved_mb = None

            if torch.cuda.is_available():
                peak_memory_allocated_mb = torch.cuda.max_memory_allocated() / 1024**2
                peak_memory_reserved_mb = torch.cuda.max_memory_reserved() / 1024**2
                peak_memory_mb = peak_memory_allocated_mb

            result.update({
                "raw_output": output_text,
                "latency_s": latency_s,
                "input_tokens": int(input_tokens),
                "output_tokens": int(output_tokens),
                "tokens_per_second": tokens_per_second,
                "peak_memory_mb": peak_memory_mb,
                "peak_memory_allocated_mb": peak_memory_allocated_mb,
                "peak_memory_reserved_mb": peak_memory_reserved_mb
            })

        elif BACKEND == "groq_prompt_api":
            messages = format_prompt(task)

            start = time.perf_counter()

            response = groq_client.chat.completions.create(
                model=config["model_name"],
                messages=messages,
                temperature=config["temperature"],
                max_tokens=config["max_new_tokens"]
            )

            latency_s = time.perf_counter() - start

            output_text = response.choices[0].message.content or ""

            usage = getattr(response, "usage", None)
            input_tokens = getattr(usage, "prompt_tokens", None) if usage else None
            output_tokens = getattr(usage, "completion_tokens", None) if usage else None
            tokens_per_second = output_tokens / latency_s if output_tokens and latency_s > 0 else None

            result.update({
                "raw_output": output_text,
                "latency_s": latency_s,
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "tokens_per_second": tokens_per_second
            })

        else:
            raise ValueError(f"Unsupported backend: {BACKEND}")

    except Exception as e:
        result["error"] = str(e)

    return result

## Test Run on Two Tasks

Before running the full benchmark, test a small subset.

In [14]:
test_results = []

for task in tasks[:2]:
    result = run_one_task(task, run_index=0)
    test_results.append(result)
    print("\nTASK:", result["task_id"], result["category"])
    print("ERROR:", result["error"])
    print("LATENCY:", result["latency_s"])
    print("OUTPUT:\n", result["raw_output"][:1000] if result["raw_output"] else None)

pd.DataFrame(test_results)

/tmp/ipykernel_6044/2432007100.py:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat()
The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



TASK: simple_361 simple
ERROR: None
LATENCY: 25.856443610000042
OUTPUT:
 <tool_call>
{"name": "restaurant_finder", "arguments": {"city": "New York", "cuisine": "Italian", "diet": "Gluten-free"}}
</tool_call>

TASK: simple_113 simple
ERROR: None
LATENCY: 26.896483095999884
OUTPUT:
 <tool_call>
{"name": "probability.dice_roll", "arguments": {"desired_number": 6, "number_of_rolls": 2, "die_sides": 6}}
</tool_call>


,task_id,category,hardware,model,backend,run_index,raw_output,latency_s,input_tokens,output_tokens,tokens_per_second,peak_memory_mb,error,timestamp
0,simple_361,simple,T4,Qwen/Qwen2.5-7B-Instruct,transformers,0,"<tool_call>\n{""name"": ""restaurant_finder"", ""ar...",25.856444,257,36,1.392303,13219.127441,None,2026-04-28T22:37:56.471539
1,simple_113,simple,T4,Qwen/Qwen2.5-7B-Instruct,transformers,0,"<tool_call>\n{""name"": ""probability.dice_roll"",...",26.896483,269,40,1.487183,13220.002441,None,2026-04-28T22:38:22.367097


## Full Benchmark Run

This writes results incrementally as JSONL so partial runs are recoverable.

In [15]:
hardware = config["hardware"]
output_path = RESULTS_DIR / f"{hardware}_results.jsonl"

print("Output path:", output_path)

Output path: results/raw/T4_results.jsonl


In [ ]:
RUN_FULL_BENCHMARK = True  # Change to True when ready

if RUN_FULL_BENCHMARK:
    with open(output_path, "a", encoding="utf-8") as f:
        for i, task in enumerate(tasks):
            print(f"[{i+1}/{len(tasks)}] Running {task['task_id']} ({task['category']})")

            result = run_one_task(task, run_index=0)

            f.write(json.dumps(result) + "\n")
            f.flush()

            print("  error:", result["error"])
            print("  latency:", result["latency_s"])
            print("  toks/sec:", result["tokens_per_second"])

    print("Benchmark complete.")
else:
    print("RUN_FULL_BENCHMARK is False. Set it to True to run all tasks.")

[1/100] Running simple_361 (simple)


/tmp/ipykernel_6044/2432007100.py:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat()


  error: None
  latency: 34.944954185999904
  toks/sec: 1.030191649655182
[2/100] Running simple_113 (simple)
  error: None
  latency: 27.598999557000298
  toks/sec: 1.4493278974619308
[3/100] Running simple_135 (simple)
  error: None
  latency: 27.1285613379996
  toks/sec: 1.4744607906638632
[4/100] Running simple_1 (simple)
  error: None
  latency: 14.107095484000183
  toks/sec: 1.488612593840988
[5/100] Running simple_123 (simple)
  error: None
  latency: 61.86466137799971
  toks/sec: 1.357802631242916
[6/100] Running simple_36 (simple)
  error: None
  latency: 23.88904654399994
  toks/sec: 1.4651066100748473
[7/100] Running simple_353 (simple)
  error: None
  latency: 21.168785629000013
  toks/sec: 1.464420328274844
[8/100] Running simple_137 (simple)
  error: None
  latency: 29.15565857899992
  toks/sec: 1.4748423495043879
[9/100] Running simple_200 (simple)
  error: None
  latency: 27.889397154000108
  toks/sec: 1.4700927299936086
[10/100] Running simple_179 (simple)
  error: Non

## Load Saved Results

In [18]:
def load_jsonl(path):
    rows = []
    if not path.exists():
        return rows

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))

    return rows

saved_rows = load_jsonl(output_path)
df_results = pd.DataFrame(saved_rows)

print("Saved rows:", len(df_results))
df_results.head()

Saved rows: 100


,task_id,category,hardware,model,backend,run_index,raw_output,latency_s,input_tokens,output_tokens,tokens_per_second,peak_memory_mb,error,timestamp
0,simple_361,simple,T4,Qwen/Qwen2.5-7B-Instruct,transformers,0,"<tool_call>\n{""name"": ""restaurant_finder"", ""ar...",34.944954,257,36,1.030192,13219.127441,None,2026-04-28T22:39:19.708422
1,simple_113,simple,T4,Qwen/Qwen2.5-7B-Instruct,transformers,0,"<tool_call>\n{""name"": ""probability.dice_roll"",...",27.599000,269,40,1.449328,13220.002441,None,2026-04-28T22:39:54.665027
2,simple_135,simple,T4,Qwen/Qwen2.5-7B-Instruct,transformers,0,"<tool_call>\n{""name"": ""calculate_return_on_inv...",27.128561,267,40,1.474461,13219.893066,None,2026-04-28T22:40:22.269049
3,simple_1,simple,T4,Qwen/Qwen2.5-7B-Instruct,transformers,0,"<tool_call>\n{""name"": ""math.factorial"", ""argum...",14.107095,179,21,1.488613,13214.038574,None,2026-04-28T22:40:49.400034
4,simple_123,simple,T4,Qwen/Qwen2.5-7B-Instruct,transformers,0,"<tool_call>\n{""name"": ""hypothesis_testing.two_...",61.864661,349,84,1.357803,13226.786621,None,2026-04-28T22:41:03.509655


In [19]:
if len(df_results) > 0:
    display(df_results.groupby("category")[["latency_s", "tokens_per_second", "peak_memory_mb"]].mean())
    display(df_results[["latency_s", "tokens_per_second", "input_tokens", "output_tokens", "peak_memory_mb"]].describe())
else:
    print("No saved rows yet.")

,latency_s,tokens_per_second,peak_memory_mb
category,,,
multiple,25.818953,1.470994,13229.573398
parallel,76.934525,1.481991,13226.530176
parallel_multiple,77.880363,1.479388,13236.872012
simple,27.982149,1.458864,13220.085566


,latency_s,tokens_per_second,input_tokens,output_tokens,peak_memory_mb
count,100.000000,100.000000,100.000000,100.000000,100.000000
mean,52.153997,1.472809,382.010000,76.900000,13228.265288
std,33.101952,0.048435,135.940903,49.016489,8.925668
min,14.107095,1.030192,179.000000,21.000000,13214.038574
25%,25.769943,1.471750,287.250000,38.000000,13222.327637
50%,39.878082,1.480033,354.000000,59.000000,13226.458984
75%,71.755593,1.485465,461.250000,107.000000,13231.492554
max,142.546014,1.502033,841.000000,210.000000,13259.178223


In [20]:
from google.colab import files
files.download("results/raw/T4_results.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>